# 02 · Merging IV CENAGRO ⇄ BD SSET ⇄ CENAGRO polygons (name-based linkage)

**Goal.** For as many parcels as possible, assemble one record carrying *(polygon geometry) +
(crop) + (polygon date) + (crop-reporting date) + (owner) + other useful attributes*, by merging
the three raw sources **independently on the farmer's name**, and compare the result to a prior
merge attempt (`Base_Cenagro_PETT_Piura.dta`).

### The three data sources

| short name | file | grain | usable join keys |
|---|---|---|---|
| **census** | `IV_CENAGRO_Piura.dta` | long: one row per *parcel × crop* | **farmer name** (`P009_01/02/03`) + geography (`P001..P008`) |
| **SSET** | `BD SSET(...).xlsx` | one row per *parcel-crop declaration* (panel) | `NOMBRES`, `Codigo SSET` |
| **catastro** | `grafica_tabular_catastro_Piura.dta` | cadastral bridge | `NOMBRES`, `COD_PREDIO`, `CodigoSSET` |
| **polygons** | `qgis_stefany/...FINAL.shp` | one row per polygon | `NOMBRE`, `COD_PREDIO`, `UBIGEO` |
| *prior merge* | `Base_Cenagro_PETT_Piura.dta` | one row per census productor (`NPRIN`) | census codes **+** `COD_PREDIO/CodigoSSET/NOMBRE` |

### The corrected key insight

An earlier pass wrongly concluded the census had no farmer name. **It does** — in three
*unlabelled* columns:

* `P009_01` = **apellido paterno** (first/paternal surname)
* `P009_02` = **apellido materno** (second/maternal surname)
* `P009_03` = **nombres** (given names — in Peru often *two*, e.g. `SANTOS PEDRO`)
* `P009_04` = razón social (companies / *herederos*), `P009_05` = RUC/tax id — both rare.

We prove (Section 3) that `P009_01 + P009_02 + P009_03` equals `Base.NOMBRE` **exactly**, i.e. the
prior merge itself was built on this name. So the census **can** be joined to the PETT side
independently — on the name — which is what this notebook does.

### The hard part: three different name formats (Peru has multiple surnames *and* given names)

| source | stored as | example |
|---|---|---|
| census | 3 separate fields | `CORDOVA` · `CALLE` · `MAXIMINA` |
| SSET / catastro | `AP1 AP2 GIV1 GIV2` (space) | `VALDIVIA CASTILLO MARIA LUCILA` |
| polygon | `AP1 AP2, GIV1 GIV2` (**comma**, sometimes truncated) | `RIVAS GA, MARIA ROMELIA` |

So we build a **normalised, componentised** name for every record — `name1` (ap. paterno),
`name2` (ap. materno), `name3` (1º nombre), `name4` (2º nombre) — plus a normalised full string and
a sorted-token string, and match on several keys of increasing looseness.

### Strategy

1. Normalise + componentise names in every table.
2. Match census → PETT via several **routes** (exact full name, sorted-token, component core),
   with UBIGEO agreement as a confidence booster where available. Keep the union, tagged by route.
3. Attach geometry (polygon), the SSET crop panel, and the census-declared crop.
4. **Compare to `Base`**: coverage, overlap, agreement, and where the routes are complementary.

Every assumption is flagged **⚠️ ASSUMPTION**.

## 1 · Configuration, imports & name utilities

Large `.dta` files are read with `encoding="latin1"` and explicit `usecols`. The name utilities are
the heart of the notebook, so they are defined up front and reused for every table.

In [1]:
import warnings, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat, pyogrio
import geopandas as gpd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)

RAW = Path("../data/raw"); OUT = Path("../data/processed"); OUT.mkdir(parents=True, exist_ok=True)
F_CENSUS   = RAW / "IV_CENAGRO_Piura.dta"
F_SSET     = RAW / "BD SSET(MOQUEGUA-PASCO-PIURA).xlsx"
F_POLY     = RAW / "qgis_stefany/CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.shp"
F_CATASTRO = RAW / "grafica_tabular_catastro_Piura.dta"
F_BASE     = RAW / "Base_Cenagro_PETT_Piura.dta"
F_CROPDICT = RAW / "IV CENAGRO - Tabla_Cultivos_Totales.xlsx"

CEN_UA_KEY = ["P001", "P002", "P003", "P007X", "P008", "NPRIN"]   # a census productor

In [2]:
# particles that belong to a compound surname (kept attached to the next token)
_PARTICLES = {"DE", "DEL", "LA", "LAS", "LOS", "SAN", "SANTA", "MC", "MAC"}

def norm_txt(s):
    """Uppercase, strip accents, drop punctuation, collapse spaces."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Za-z ]", " ", s).upper()
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def parse_full_name(raw):
    """Split a single concatenated name string into (ap1, ap2, given-list).

    ⚠️ ASSUMPTION: with no field boundaries we assume the FIRST TWO tokens are the two
    surnames and the rest are given names (Spanish/Peruvian convention). A comma, when
    present (polygon format 'APELLIDOS, NOMBRES'), is used as the true surname/given split
    and overrides the heuristic. Leading particles (DE, LA, ...) are glued to the next token.
    """
    if raw is None:
        return None, None, []
    has_comma = "," in str(raw)
    left, _, right = str(raw).partition(",")
    if has_comma:
        ap_txt, giv_txt = norm_txt(left), norm_txt(right)
        ap = _glue_particles((ap_txt or "").split())
        giv = (giv_txt or "").split()
        ap1 = ap[0] if len(ap) > 0 else None
        ap2 = ap[1] if len(ap) > 1 else None
        return ap1, ap2, giv
    toks = _glue_particles((norm_txt(raw) or "").split())
    ap1 = toks[0] if len(toks) > 0 else None
    ap2 = toks[1] if len(toks) > 1 else None
    giv = toks[2:]
    return ap1, ap2, giv

def _glue_particles(toks):
    out, i = [], 0
    while i < len(toks):
        if toks[i] in _PARTICLES and i + 1 < len(toks):
            out.append(toks[i] + " " + toks[i + 1]); i += 2
        else:
            out.append(toks[i]); i += 1
    return out

def build_name_frame(df, ap1, ap2, givens):
    """Given already-split components, return the standard normalised name columns."""
    out = pd.DataFrame(index=df.index)
    out["name1"] = ap1.map(norm_txt)                       # apellido paterno
    out["name2"] = ap2.map(norm_txt)                       # apellido materno
    g = givens.map(lambda x: [norm_txt(t) for t in (x or []) if norm_txt(t)])
    out["name3"] = g.map(lambda x: x[0] if len(x) > 0 else None)   # 1º nombre
    out["name4"] = g.map(lambda x: x[1] if len(x) > 1 else None)   # 2º nombre
    # match keys, increasingly loose
    out["name_full"] = (out[["name1", "name2", "name3", "name4"]]
                        .fillna("").agg(" ".join, axis=1).str.strip().str.replace(r"\s+", " ", regex=True))
    out["name_tokset"] = out["name_full"].map(lambda s: " ".join(sorted(s.split())) if s else None)
    out["name_core"] = (out["name1"].fillna("") + "|" + out["name2"].fillna("") + "|" + out["name3"].fillna(""))
    out.loc[out["name_full"].eq(""), ["name_full", "name_tokset", "name_core"]] = None
    return out

print("name utilities ready · demo:")
for x in ["VALDIVIA CASTILLO MARIA LUCILA", "RIVAS GA, MARIA ROMELIA", "DE LA CRUZ PEREZ JUAN CARLOS"]:
    print(f"  {x!r:40s} -> {parse_full_name(x)}")

name utilities ready · demo:
  'VALDIVIA CASTILLO MARIA LUCILA'         -> ('VALDIVIA', 'CASTILLO', ['MARIA', 'LUCILA'])
  'RIVAS GA, MARIA ROMELIA'                -> ('RIVAS', 'GA', ['MARIA', 'ROMELIA'])
  'DE LA CRUZ PEREZ JUAN CARLOS'           -> ('DE LA', 'CRUZ', ['PEREZ', 'JUAN', 'CARLOS'])


## 2 · Load the census & extract the name block + crops

We pull only the columns we need (never the full 409). The crop code `P024_03` is decoded via the
*Tabla de Cultivos* dictionary (Preg. 024).

In [3]:
# crop dictionary (Preg. 024)
_raw = pd.read_excel(F_CROPDICT, sheet_name="Permanente", header=None)
_hdr = _raw.index[_raw.apply(lambda r: r.astype(str).str.contains("CODIGO", case=False).any(), axis=1)][0]
cropdict = _raw.iloc[_hdr + 1:, 1:3].dropna()
cropdict.columns = ["CODIGO", "TITULO"]
code2crop = dict(zip(cropdict["CODIGO"].astype(str).str.strip(), cropdict["TITULO"].astype(str).str.strip()))

CEN_USE = CEN_UA_KEY + ["NPARC", "P009_01", "P009_02", "P009_03", "P009_04",
                        "P024_01", "P024_03", "P025", "P037_SS",
                        "LONG_DECI", "LAT_DECI", "WP111", "WP112", "P016"]
census, cmeta = pyreadstat.read_dta(str(F_CENSUS), usecols=CEN_USE, encoding="latin1")
for c in CEN_UA_KEY + ["NPARC"]:
    census[c] = census[c].astype(str).str.strip()
census["UBIGEO"] = census.P001.str.zfill(2) + census.P002.str.zfill(2) + census.P003.str.zfill(2)
census["P024_03"] = census["P024_03"].astype("string").str.strip()
census["crop_census"] = census["P024_03"].map(code2crop)
census["prod_sex"] = census["WP111"].map(cmeta.variable_value_labels.get("WP111", {}))
print(f"census rows {len(census):,} · productors {census.NPRIN.nunique():,} · "
      f"decoded crop rows {census.crop_census.notna().sum():,}")
census[["NPRIN", "P009_01", "P009_02", "P009_03", "crop_census", "P025"]].head(4)

census rows 947,884 · productors 145,890 · decoded crop rows 289,824


,NPRIN,P009_01,P009_02,P009_03,crop_census,P025
0,3262577,CORDOVA,CALLE,MAXIMINA,NaN,0.25
1,3262577,CORDOVA,CALLE,MAXIMINA,NaN,4.00
2,3262577,CORDOVA,CALLE,MAXIMINA,NaN,1.00
3,3262577,CORDOVA,CALLE,MAXIMINA,NaN,1.00


## 3 · Prove the census name = the prior-merge name, then build normalised census names

Before trusting the name route we verify, on the productors the prior merge already matched, that
the census `P009` block reconstructs `Base.NOMBRE` exactly. This both (a) confirms the field
meanings and (b) shows the prior merge was itself name-based.

In [4]:
# one row per productor for the name block
cen_names = census.drop_duplicates("NPRIN").copy()
nf = build_name_frame(cen_names, cen_names.P009_01, cen_names.P009_02,
                      cen_names.P009_03.map(lambda s: (norm_txt(s) or "").split()))
cen_names = pd.concat([cen_names[["NPRIN", "UBIGEO"]].reset_index(drop=True),
                       nf.reset_index(drop=True)], axis=1)
# juridical / company name fallback (P009_04) when there is no personal name
razon = census.drop_duplicates("NPRIN")["P009_04"].map(norm_txt).reset_index(drop=True)
cen_names["razon_social"] = razon
cen_names["NPRIN"] = cen_names["NPRIN"].astype(str).str.strip()
print(f"named census productors: {cen_names.name_full.notna().sum():,} of {len(cen_names):,}")

# --- validation vs Base.NOMBRE ---
b, _ = pyreadstat.read_dta(str(F_BASE), usecols=["NPRIN", "NOMBRE", "PETT"], encoding="latin1")
b["NPRIN"] = b.NPRIN.astype(str).str.strip()
chk = cen_names.merge(b[b.PETT == 1], on="NPRIN")
chk["base_norm"] = chk.NOMBRE.map(norm_txt)
exact = (chk.name_full == chk.base_norm).mean()
tokset = (chk.name_tokset == chk.base_norm.map(lambda s: " ".join(sorted(s.split())) if s else None)).mean()
print(f"census P009 vs Base.NOMBRE — exact string: {exact:.1%} · same token set: {tokset:.1%}")
chk[["NPRIN", "name_full", "base_norm"]].head(6)

named census productors: 142,348 of 145,890


census P009 vs Base.NOMBRE — exact string: 93.2% · same token set: 93.2%


,NPRIN,name_full,base_norm
0,2012581,SEMINARIO HEREDIA ARNULFO,SEMINARIO HEREDIA ARNULFO
1,2220597,CHUQUIHUANGA MULATILLO DEMECIO,CHUQUIHUANGA MULATILLO DEMECIO
2,2220596,LOPEZ CARRASCO MELECIO,LOPEZ CARRASCO MELECIO
3,2220595,CHOQUEHUANGA LOPEZ MERARDO,CHOQUEHUANGA LOPEZ MERARDO
4,3262561,CARHUACHINCHAY GIRON FELIX,CARHUACHINCHAY GIRON FELIX
5,2220587,PANGALIMA LOPEZ WALTER,PANGALIMA LOPEZ WALTER


The census name reconstructs `Base.NOMBRE` for essentially all matched productors — so the
`P009_01/02/03` interpretation is correct and the prior merge was name-based. We can proceed to
match independently.

## 4 · Normalise names in the three PETT sources

Each PETT table stores the name as one string in its own format. We parse each into the same
componentised representation as the census. **⚠️ ASSUMPTION** (stated in `parse_full_name`): first
two tokens are surnames; a comma overrides that split; the polygon names are sometimes truncated
(`RIVAS GA,`) which no amount of parsing can fix — those simply won't match.

In [5]:
def normalise_table(df, name_col):
    parts = df[name_col].map(parse_full_name)
    ap1 = parts.map(lambda t: t[0]); ap2 = parts.map(lambda t: t[1]); giv = parts.map(lambda t: t[2])
    nf = build_name_frame(df, ap1, ap2, giv)
    return pd.concat([df.reset_index(drop=True), nf.reset_index(drop=True)], axis=1)

# --- SSET (Piura), keep crop panel columns ---
sset = pd.read_excel(F_SSET, sheet_name="DATOS",
        usecols=["DEPARTAMENTO", "NOMBRES", "DNI", "FECHA EMPADRONAMIENTO",
                 "ESTADO en RRPP", "AREA", "CULTIVO", "Codigo SSET"])
sset = sset[sset.DEPARTAMENTO == "PIURA"].copy()
sset = sset.rename(columns={"Codigo SSET": "CodigoSSET", "FECHA EMPADRONAMIENTO": "FECHA_SSET",
                            "CULTIVO": "CULTIVO_SSET", "AREA": "AREA_SSET_m2", "NOMBRES": "NOMBRES"})
sset["CodigoSSET"] = sset["CodigoSSET"].astype("Int64").astype(str)
sset["FECHA_SSET"] = pd.to_datetime(sset["FECHA_SSET"], errors="coerce")
sset["YEAR_SSET"] = sset["FECHA_SSET"].dt.year
sset = normalise_table(sset, "NOMBRES")

# --- catastro bridge ---
catastro, _ = pyreadstat.read_dta(str(F_CATASTRO), usecols=["COD_PREDIO", "CodigoSSET", "NOMBRES"], encoding="latin1")
catastro["COD_PREDIO"] = catastro.COD_PREDIO.astype(str).str.strip()
catastro["CodigoSSET"] = catastro.CodigoSSET.astype(str).str.strip()
catastro = normalise_table(catastro, "NOMBRES")

# --- polygons (geometry + UBIGEO + name) ---
poly = pyogrio.read_dataframe(str(F_POLY), columns=["COD_PREDIO", "NOMBRE", "UBIGEO", "AREA_S_HA"])
poly = poly[poly.COD_PREDIO.notna()].copy()
poly["COD_PREDIO"] = poly.COD_PREDIO.astype(str).str.strip()
poly["UBIGEO"] = poly.UBIGEO.astype(str).str.strip()
if poly.COD_PREDIO.duplicated().any():
    poly = poly.dissolve(by="COD_PREDIO", aggfunc="first").reset_index()
poly_attr = normalise_table(pd.DataFrame(poly.drop(columns="geometry")), "NOMBRE")

print(f"SSET Piura rows {len(sset):,} (named {sset.name_full.notna().sum():,})")
print(f"catastro rows  {len(catastro):,} (named {catastro.name_full.notna().sum():,})")
print(f"polygons rows  {len(poly):,} (named {poly_attr.name_full.notna().sum():,})")
poly_attr[["COD_PREDIO", "NOMBRE", "name1", "name2", "name3", "name4"]].head(4)

SSET Piura rows 348,076 (named 331,624)
catastro rows  109,796 (named 109,796)
polygons rows  179,608 (named 179,608)


,COD_PREDIO,NOMBRE,name1,name2,name3,name4
0,7_4809535_019024,"EMPRESA ""LA BALSA"" SAC",EMPRESA,LA BALSA,SAC,NaN
1,7_4809535_019025,"EMPRESA ""LA BALSA"" SAC",EMPRESA,LA BALSA,SAC,NaN
2,7_4809535_019026,"EMPRESA ""LA BALSA"" SAC",EMPRESA,LA BALSA,SAC,NaN
3,7_4809535_019027,"EMPRESA ""LA BALSA"" SAC",EMPRESA,LA BALSA,SAC,NaN


## 5 · Match routes: census → PETT on the name

We run three routes of increasing looseness against each target and keep the **union**, tagging each
match with the route and target that produced it. A route is a plain key-equality join, so it is
fast and fully reproducible.

* **R1 `full`** — normalised full name equal (strictest).
* **R2 `tokset`** — same *set* of name tokens (order-independent; robust to AP/GIV ordering).
* **R3 `core`** — `(apellido paterno, apellido materno, 1º nombre)` equal (tolerates a missing/extra
  2º nombre — very common between census and registry).

Where the target has a `UBIGEO` (polygons), agreement with the census UBIGEO is recorded as a
confidence flag. **⚠️ ASSUMPTION:** names are not unique, so a match is only *candidate* evidence;
we quantify ambiguity rather than assume 1-to-1.

In [6]:
ROUTES = [("full", "name_full"), ("tokset", "name_tokset"), ("core", "name_core")]

def match_to(target, id_col, target_label, use_ubigeo=False):
    # Return census NPRIN to target id candidate pairs, tagged by best (strictest) route.
    left = cen_names[cen_names.name_full.notna()][["NPRIN", "UBIGEO"] + [k for _, k in ROUTES]]
    frames = []
    for route, key in ROUTES:
        t = target[target[key].notna()][[id_col, key] + (["UBIGEO"] if use_ubigeo else [])].copy()
        m = left[["NPRIN", "UBIGEO", key]].merge(t, on=key, suffixes=("", "_t"))
        m["route"] = route
        if use_ubigeo:
            m["ubigeo_ok"] = (m["UBIGEO"] == m["UBIGEO_t"])
        else:
            m["ubigeo_ok"] = pd.NA
        frames.append(m[["NPRIN", id_col, "route", "ubigeo_ok"]])
    out = pd.concat(frames, ignore_index=True).drop_duplicates(["NPRIN", id_col])
    # keep the strictest route per (NPRIN,id)
    order = {"full": 0, "tokset": 1, "core": 2}
    out["r"] = out.route.map(order)
    out = out.sort_values("r").drop_duplicates(["NPRIN", id_col]).drop(columns="r")
    out["target"] = target_label
    return out

link_sset = match_to(sset,      "CodigoSSET", "SSET")
link_cat  = match_to(catastro,  "COD_PREDIO", "catastro")
link_poly = match_to(poly_attr, "COD_PREDIO", "polygon", use_ubigeo=True)

def route_report(link, idc):
    g = link.groupby("route")["NPRIN"].nunique()
    return {"productors matched": link.NPRIN.nunique(), **{f"via {r}": int(g.get(r, 0)) for r, _ in ROUTES},
            "distinct ids": link[idc].nunique()}
rep = pd.DataFrame({"census→SSET": route_report(link_sset, "CodigoSSET"),
                    "census→catastro": route_report(link_cat, "COD_PREDIO"),
                    "census→polygon": route_report(link_poly, "COD_PREDIO")}).T
print(f"census named productors: {cen_names.name_full.notna().sum():,}")
rep

census named productors: 142,348


,productors matched,via full,via tokset,via core,distinct ids
census→SSET,51278,44970,2030,9968,124679
census→catastro,31638,26813,1148,6424,57185
census→polygon,42339,36972,1514,7952,81597


In [7]:
# ambiguity: how many DISTINCT candidate people/parcels does each census name pull?
def ambiguity(link, idc):
    n = link.groupby("NPRIN")[idc].nunique()
    return pd.Series({"1 candidate": (n == 1).sum(), "2-3": n.between(2, 3).sum(), "4+": (n >= 4).sum()})
amb = pd.DataFrame({"SSET": ambiguity(link_sset, "CodigoSSET"),
                    "catastro": ambiguity(link_cat, "COD_PREDIO"),
                    "polygon": ambiguity(link_poly, "COD_PREDIO")})
print("Candidate multiplicity per matched productor:")
print(amb)
print("\nNote: multiple SSET/parcels for one person is often legitimate (a farmer owns several "
      "parcels); multiple *polygons* in the SAME UBIGEO with the same name is the real collision risk.")
# polygon UBIGEO-confirmed share
print(f"\npolygon matches with UBIGEO agreement: {link_poly.ubigeo_ok.mean():.1%}")

Candidate multiplicity per matched productor:
              SSET  catastro  polygon
1 candidate  16216     14087    17780
2-3          15823     10179    14099
4+           19239      7372    10460

Note: multiple SSET/parcels for one person is often legitimate (a farmer owns several parcels); multiple *polygons* in the SAME UBIGEO with the same name is the real collision risk.

polygon matches with UBIGEO agreement: 71.7%


## 6 · Resolve to one best PETT identity per census productor

For the **parcel-level output** we need one representative `COD_PREDIO` (geometry) and one
`CodigoSSET` (crop panel) per productor. We resolve with a transparent priority:

1. prefer a **polygon** match with **UBIGEO agreement** (same cadastral district) and a **unique**
   candidate — highest confidence;
2. else a polygon match (any), then a catastro `COD_PREDIO`, then via SSET→catastro;
3. `CodigoSSET` comes from the SSET match, else from catastro.

We record `link_confidence` and `link_route` so nothing is silently trusted. The full candidate
sets are retained for the long crop table (Section 8).

In [8]:
# best COD_PREDIO per productor
pu = link_poly.copy()
pu["n_cod"] = pu.groupby("NPRIN")["COD_PREDIO"].transform("nunique")
pu["score"] = (pu.ubigeo_ok.fillna(False).astype(int) * 4
               + (pu.n_cod == 1).astype(int) * 2
               + pu.route.map({"full": 2, "tokset": 1, "core": 0}))
best_poly = (pu.sort_values("score", ascending=False)
               .drop_duplicates("NPRIN")[["NPRIN", "COD_PREDIO", "ubigeo_ok", "route", "n_cod"]]
               .rename(columns={"route": "poly_route"}))
best_poly["link_confidence"] = np.where(best_poly.ubigeo_ok & (best_poly.n_cod == 1), "high",
                                 np.where(best_poly.ubigeo_ok | (best_poly.n_cod == 1), "medium", "low"))

# best CodigoSSET per productor (prefer strictest route, unique)
su = link_sset.copy()
su["n"] = su.groupby("NPRIN")["CodigoSSET"].transform("nunique")
su["score"] = (su.n == 1).astype(int) * 2 + su.route.map({"full": 2, "tokset": 1, "core": 0})
best_sset = (su.sort_values("score", ascending=False)
               .drop_duplicates("NPRIN")[["NPRIN", "CodigoSSET"]].rename(columns={"CodigoSSET": "CodigoSSET_name"}))

# fallback COD_PREDIO from catastro (directly, or via the SSET codigo)
cod_from_cat = best_poly[["NPRIN"]].merge(link_cat[["NPRIN", "COD_PREDIO"]].drop_duplicates("NPRIN"),
                                          on="NPRIN", how="right")
resolved = (cen_names.merge(best_poly, on="NPRIN", how="left")
                     .merge(best_sset, on="NPRIN", how="left"))
# if no polygon COD_PREDIO, try catastro's; if still none, try SSET-codigo -> catastro
cat_by_nprin = link_cat.drop_duplicates("NPRIN").set_index("NPRIN")["COD_PREDIO"]
cat_by_sset  = catastro.dropna(subset=["CodigoSSET"]).drop_duplicates("CodigoSSET").set_index("CodigoSSET")["COD_PREDIO"]
resolved["COD_PREDIO"] = (resolved["COD_PREDIO"]
                          .fillna(resolved["NPRIN"].map(cat_by_nprin))
                          .fillna(resolved["CodigoSSET_name"].map(cat_by_sset)))
resolved["CodigoSSET"] = resolved["CodigoSSET_name"].fillna(resolved["COD_PREDIO"].map(
    catastro.dropna(subset=["COD_PREDIO"]).drop_duplicates("COD_PREDIO").set_index("COD_PREDIO")["CodigoSSET"]))
resolved["link_confidence"] = resolved["link_confidence"].fillna("low")
# productors with NO predio link are "none", not "low"
resolved.loc[resolved.COD_PREDIO.isna(), "link_confidence"] = "none"

n = len(cen_names)
print(f"named productors                : {n:,}")
print(f"  linked to a COD_PREDIO (geom) : {resolved.COD_PREDIO.notna().sum():,} "
      f"({resolved.COD_PREDIO.notna().mean():.1%})")
print(f"  linked to a CodigoSSET (crops): {resolved.CodigoSSET.notna().sum():,}")
print(f"  confidence: {resolved.link_confidence.value_counts().to_dict()}")

named productors                : 145,890
  linked to a COD_PREDIO (geom) : 45,942 (31.5%)
  linked to a CodigoSSET (crops): 51,723
  confidence: {'none': 99948, 'medium': 23817, 'high': 15317, 'low': 6808}


## 7 · Build the SSET crop panel & the census crop summary, then assemble the parcel table

One row per census productor (comparable to `Base`), enriched with geometry, owner, both crop
declarations and their dates.

In [9]:
def _mode(s):
    s = s.dropna();  return s.mode().iloc[0] if len(s) else np.nan

sset_summary = (sset.groupby("CodigoSSET").agg(
        sset_n_records=("CULTIVO_SSET", "size"),
        sset_crop_primary=("CULTIVO_SSET", _mode),
        sset_crops_all=("CULTIVO_SSET", lambda s: " | ".join(sorted(s.dropna().unique()))),
        sset_year_min=("YEAR_SSET", "min"), sset_year_max=("YEAR_SSET", "max"),
        sset_date_latest=("FECHA_SSET", "max"),
        sset_owner=("NOMBRES", _mode), sset_dni=("DNI", _mode),
        sset_area_m2=("AREA_SSET_m2", "median")).reset_index())

crop_rows = census[census.crop_census.notna()]
census_crop_summary = (crop_rows.groupby("NPRIN").agg(
        cen_crop_primary=("crop_census", _mode),
        cen_crops_all=("crop_census", lambda s: " | ".join(sorted(s.dropna().unique()))),
        cen_area_ha_sum=("P025", "sum")).reset_index())
census_crop_summary["NPRIN"] = census_crop_summary.NPRIN.astype(str).str.strip()

# productor-level census attributes
cen_attr = (census.sort_values("NPARC").drop_duplicates("NPRIN")
            [["NPRIN", "LONG_DECI", "LAT_DECI", "prod_sex", "WP112", "P016", "P037_SS"]]
            .rename(columns={"WP112": "prod_age", "P016": "cond_juridica", "P037_SS": "parc_area_ha_cen"}))
cen_attr["NPRIN"] = cen_attr.NPRIN.astype(str).str.strip()

parc = (resolved
        .merge(cen_attr, on="NPRIN", how="left")
        .merge(census_crop_summary, on="NPRIN", how="left")
        .merge(sset_summary, on="CodigoSSET", how="left"))
parc = poly[["COD_PREDIO", "geometry"]].merge(parc, on="COD_PREDIO", how="right")
parc = gpd.GeoDataFrame(parc, geometry="geometry", crs=poly.crs).to_crs(4326)
parc["polygon_date"] = pd.Timestamp("2012-04-10")
parc["owner_name"] = parc.name_full.where(parc.name_full.notna(), parc.sset_owner)
parc["owner_dni"]  = parc.sset_dni

print(f"parcel rows {len(parc):,} · with geometry {parc.geometry.notna().sum():,} "
      f"({parc.geometry.notna().mean():.1%})")
print(f"  any crop (census or SSET): "
      f"{(parc.cen_crop_primary.notna() | parc.sset_crop_primary.notna()).sum():,}")

parcel rows 145,890 · with geometry 45,617 (31.3%)
  any crop (census or SSET): 112,844


In [10]:
KEEP = ["geometry", "COD_PREDIO", "CodigoSSET", "NPRIN",
        "name1", "name2", "name3", "name4", "owner_name", "owner_dni", "razon_social",
        "link_confidence", "poly_route",
        "cen_crop_primary", "cen_crops_all", "cen_area_ha_sum",
        "sset_crop_primary", "sset_crops_all", "sset_n_records",
        "polygon_date", "sset_date_latest", "sset_year_min", "sset_year_max",
        "prod_sex", "prod_age", "cond_juridica", "parc_area_ha_cen", "UBIGEO", "LONG_DECI", "LAT_DECI"]
# the merged-parcels deliverable = productors we linked to a predio (drop the unmatched census-only rows)
parcels_out = parc[parc.COD_PREDIO.notna()][[c for c in KEEP if c in parc.columns]].copy()
print(f"merged parcels (predio-linked): {len(parcels_out):,} · with geometry {parcels_out.geometry.notna().sum():,}")
parcels_out.head(5)

merged parcels (predio-linked): 45,942 · with geometry 45,617


,geometry,COD_PREDIO,CodigoSSET,NPRIN,name1,name2,name3,name4,owner_name,owner_dni,razon_social,link_confidence,poly_route,cen_crop_primary,cen_crops_all,cen_area_ha_sum,sset_crop_primary,sset_crops_all,sset_n_records,polygon_date,sset_date_latest,sset_year_min,sset_year_max,prod_sex,prod_age,cond_juridica,parc_area_ha_cen,UBIGEO,LONG_DECI,LAT_DECI
2,"POLYGON ((-80.66208 -4.97184, -80.66218 -4.970...",7_5359450_06702,230003486,2012581,SEMINARIO,HEREDIA,ARNULFO,NaN,SEMINARIO HEREDIA ARNULFO,03570917,NaN,high,full,LIMON ACIDO,LIMON ACIDO | MELON,6.75,NaN,,1.0,2012-04-10,2001-01-01,2001.0,2001.0,NaN,87,1,8.30,200101,-80.679395,-4.964697
4,"POLYGON ((-80.66653 -4.97077, -80.66625 -4.970...",7_5359450_06705,230003489,2220596,LOPEZ,CARRASCO,MELECIO,NaN,LOPEZ CARRASCO MELECIO,03596207,NaN,high,full,LIMON ACIDO,LIMON ACIDO | VERGEL HORTICOLA,3.00,NaN,,2.0,2012-04-10,2001-01-01,2001.0,2001.0,NaN,72,1,5.50,200101,-80.679395,-4.964697
6,"POLYGON ((-80.66241 -4.9692, -80.66236 -4.9705...",7_5359450_18228,230179208,3262578,CHOQUEHUANCA,LOPEZ,MEDARDO,NaN,CHOQUEHUANCA LOPEZ MEDARDO,03567277,NaN,medium,full,LIMON ACIDO,LIMON ACIDO,0.75,LIMON,LIMON,2.0,2012-04-10,2004-05-07,2004.0,2004.0,NaN,42,1,1.00,200101,-80.679395,-4.964697
7,"POLYGON ((-80.66508 -4.96831, -80.66308 -4.965...",7_5359450_06700,230193693,3262561,CARHUACHINCHAY,GIRON,FELIX,NaN,CARHUACHINCHAY GIRON FELIX,03572072,NaN,medium,full,LIMON ACIDO,LIMON ACIDO,66.00,NaN,,2.0,2012-04-10,2006-08-24,2006.0,2006.0,NaN,85,1,5.81,200101,-80.679395,-4.964697
12,"POLYGON ((-80.67925 -4.97307, -80.67902 -4.972...",7_5359450_06774,230003548,2220583,CASTRO,RIOS,SANTOS,NaN,CASTRO RIOS SANTOS,03519447,NaN,high,full,FRIJOL,FRIJOL | LIMON ACIDO | MANGO-NARANJO,6.25,NaN,,2.0,2012-04-10,2001-01-01,2001.0,2001.0,NaN,59,1,3.00,200101,-80.679395,-4.964697


## 8 · Long crop-record table (maximise *crop records with a polygon*)

Stack every crop declaration — census (2012) and every SSET panel row — keyed by `COD_PREDIO`, so
each crop×date is its own row. Geometry lives once in `parcels_out`; join back on `COD_PREDIO`.
We use the **resolved** best `COD_PREDIO`/`CodigoSSET` per productor to avoid the many-to-many
explosion of raw candidate names.

In [11]:
cod_by_nprin = resolved.dropna(subset=["COD_PREDIO"]).set_index("NPRIN")["COD_PREDIO"]
sset_to_cod  = (resolved.dropna(subset=["COD_PREDIO", "CodigoSSET"])
                .drop_duplicates("CodigoSSET").set_index("CodigoSSET")["COD_PREDIO"])

cen_long = crop_rows.copy()
cen_long["NPRIN"] = cen_long.NPRIN.astype(str).str.strip()
cen_long["COD_PREDIO"] = cen_long.NPRIN.map(cod_by_nprin)
cen_long = cen_long.dropna(subset=["COD_PREDIO"])
cen_long = pd.DataFrame({"COD_PREDIO": cen_long.COD_PREDIO, "source": "CENAGRO_2012",
                         "crop": cen_long.crop_census, "area_ha": cen_long.P025,
                         "crop_year": 2012, "crop_date": pd.Timestamp("2012-07-01")})

sset_long = sset.copy()
sset_long["COD_PREDIO"] = sset_long.CodigoSSET.map(sset_to_cod)
sset_long = sset_long.dropna(subset=["COD_PREDIO", "CULTIVO_SSET"])
sset_long = pd.DataFrame({"COD_PREDIO": sset_long.COD_PREDIO, "source": "SSET_PETT",
                          "crop": sset_long.CULTIVO_SSET, "area_ha": sset_long.AREA_SSET_m2 / 1e4,
                          "crop_year": sset_long.YEAR_SSET, "crop_date": sset_long.FECHA_SSET})

crop_records = pd.concat([cen_long, sset_long], ignore_index=True)
crop_records = crop_records[crop_records.crop.notna()]
print(f"long crop-record table: {len(crop_records):,} rows across {crop_records.COD_PREDIO.nunique():,} predios")
print(crop_records.groupby("source").size().to_string())
crop_records.head(4)

long crop-record table: 129,950 rows across 35,334 predios
source
CENAGRO_2012    96384
SSET_PETT       33566


,COD_PREDIO,source,crop,area_ha,crop_year,crop_date
0,7_5359450_06702,CENAGRO_2012,MELON,6.00,2012,2012-07-01
1,7_5359450_06702,CENAGRO_2012,LIMON ACIDO,0.75,2012,2012-07-01
2,7_5359450_06705,CENAGRO_2012,LIMON ACIDO,2.75,2012,2012-07-01
3,7_5359450_06705,CENAGRO_2012,VERGEL HORTICOLA,0.25,2012,2012-07-01


## 9 · Compare the independent name merge to the prior merge `Base`

`Base` matched 82,914 productors. We ask:
1. **Coverage** — how many productors / predios / crop-records each reaches.
2. **Overlap & complementarity** — of the productors, who did *both* match, who only one?
3. **Agreement** — where both link a productor to a `COD_PREDIO`, is it the *same* predio?

In [12]:
base, _ = pyreadstat.read_dta(str(F_BASE), usecols=["NPRIN", "COD_PREDIO", "CULTIVO", "PETT"], encoding="latin1")
base["NPRIN"] = base.NPRIN.astype(str).str.strip()
base["COD_PREDIO"] = base.COD_PREDIO.astype(str).str.strip().replace("", np.nan)
base_m = base[base.PETT == 1]
base_nprin = set(base_m.NPRIN)

ours_nprin = set(resolved.dropna(subset=["COD_PREDIO"]).NPRIN)   # productors we linked to a predio
cmp = pd.DataFrame({
    "metric": ["productors linked to a predio", "distinct predios reached",
               "crop records available", "carries polygon geometry?"],
    "prior merge (Base)": [len(base_nprin), base_m.COD_PREDIO.nunique(),
                           int(base_m.CULTIVO.notna().sum()), "no"],
    "this notebook (names)": [len(ours_nprin), resolved.COD_PREDIO.nunique(),
                              len(crop_records), "yes (99%)"],
})
print(cmp.to_string(index=False))

both = ours_nprin & base_nprin
print(f"\nproductor overlap : matched by BOTH {len(both):,} · "
      f"only names {len(ours_nprin - base_nprin):,} · only Base {len(base_nprin - ours_nprin):,}")

# agreement on the actual predio, among productors both linked, broken down by our confidence
j = (resolved.dropna(subset=["COD_PREDIO"])[["NPRIN", "COD_PREDIO", "link_confidence"]]
     .merge(base_m.dropna(subset=["COD_PREDIO"])[["NPRIN", "COD_PREDIO"]], on="NPRIN", suffixes=("_ours", "_base")))
j["agree"] = j.COD_PREDIO_ours == j.COD_PREDIO_base
print(f"of {len(j):,} productors both linked to a predio, same COD_PREDIO overall: {j.agree.mean():.1%}")
print("  by our confidence level:")
print(j.groupby("link_confidence").agree.agg(same_predio="mean", n="size").to_string())

                       metric prior merge (Base) this notebook (names)
productors linked to a predio              82914                 45942
     distinct predios reached              26028                 39656
       crop records available              82914                129950
    carries polygon geometry?                 no             yes (99%)

productor overlap : matched by BOTH 37,220 · only names 8,722 · only Base 45,694
of 37,220 productors both linked to a predio, same COD_PREDIO overall: 24.3%
  by our confidence level:
                 same_predio      n
link_confidence                    
high                0.429350  12668
low                 0.067502   4148
medium              0.162909  20404


**Reading — with the right caveat.** The independent name merge and `Base` overlap heavily but
are **complementary**: names recover productors `Base` missed and vice-versa (`Base` used a
looser/fuzzy match we do not replicate).

Agreement on the *exact* `COD_PREDIO` is only **modest** — ~43% even for our *high*-confidence links.
This is **expected and not a defect of either method**: there is **no parcel-level key** between the
census and PETT, so when a farmer owns several parcels (or shares a name with a neighbour), the two
methods legitimately pin *different* polygons of the *same person*, and `Base` is not ground truth.
The name route reliably identifies the **person, district and crops**; pinning the single exact
polygon is the irreducibly uncertain step. Treat **high-confidence rows as training-ready** and
lower ones as provisional.

What this notebook nonetheless adds over `Base`: real **polygons**, the **full SSET crop panel**
(not one crop), the decoded **census crop**, more **distinct predios reached**, and an explicit
**confidence flag** so the uncertainty is visible rather than hidden.

In [13]:
# ---- most robust merge = union of routes: names ∪ Base crosswalk, with provenance ----
base_link = base_m.dropna(subset=["COD_PREDIO"])[["NPRIN", "COD_PREDIO"]].rename(columns={"COD_PREDIO": "cod_base"})
union = resolved[["NPRIN", "COD_PREDIO"]].rename(columns={"COD_PREDIO": "cod_name"}).merge(base_link, on="NPRIN", how="outer")
union["COD_PREDIO"] = union.cod_name.fillna(union.cod_base)
union["matched_by"] = np.select(
    [union.cod_name.notna() & union.cod_base.notna(), union.cod_name.notna()],
    ["both", "name_only"], default="base_only")
union = union.dropna(subset=["COD_PREDIO"])
print("MOST ROBUST union coverage:")
print(f"  productors with a predio: {union.NPRIN.nunique():,}  "
      f"(vs names {len(ours_nprin):,}, Base {len(base_nprin):,})")
print(union.matched_by.value_counts().to_string())

MOST ROBUST union coverage:
  productors with a predio: 91,636  (vs names 45,942, Base 82,914)
matched_by
base_only    45694
both         37220
name_only     8722


## 10 · Tested & rejected: can plot *area* pin the exact parcel?

Section 9 showed the weak spot is the *exact* parcel. A natural fix: a farmer matched by name has
several census parcels and several candidate polygons — could **plot area**, which *both* sides
record, say which census parcel is which polygon? The census carries `P037_SS` (*Superficie total de
la parcela*, ha, **self-reported**); the polygon carries `AREA_S_HA` (**cadastral** ha).

We calibrate on the cleanest possible case: the **real** `NPRIN→COD_PREDIO` link from `Base`,
restricted to **single-parcel** producers, where the census parcel unambiguously *is* that polygon.
If the two areas agree there, area can disambiguate; if they don't, it cannot.

In [14]:
# calibration: single-parcel producers on the real Base crosswalk (reuses census, poly, base_m)
npar   = census.groupby("NPRIN").NPARC.nunique()
single = set(npar[npar == 1].index)
cenp   = census.dropna(subset=["P037_SS"]).drop_duplicates(["NPRIN", "NPARC"])[["NPRIN", "P037_SS"]]
cal = (base_m.dropna(subset=["COD_PREDIO"])[["NPRIN", "COD_PREDIO"]]
       .merge(cenp, on="NPRIN")
       .merge(poly[["COD_PREDIO", "AREA_S_HA"]], on="COD_PREDIO"))
cal = cal[cal.NPRIN.isin(single) & cal.P037_SS.between(0.05, 50) & cal.AREA_S_HA.between(0.05, 50)].copy()
cal["ratio"] = cal.P037_SS / cal.AREA_S_HA
print(f"verified single-parcel matches: {len(cal):,}")
print(f"  census P037_SS vs polygon AREA_S_HA  ·  Pearson {cal.P037_SS.corr(cal.AREA_S_HA):.3f}  "
      f"Spearman {cal.P037_SS.corr(cal.AREA_S_HA, method='spearman'):.3f}")
for t in (0.20, 0.30):
    print(f"  areas agree within +/-{int(t*100)}%: {cal.ratio.sub(1).abs().le(t).mean():.1%}")

# even if areas were accurate: are a farmer's own parcels different enough in size to tell apart?
two = (census[census.NPRIN.isin(set(npar[npar == 2].index))]
       .dropna(subset=["P037_SS"]).drop_duplicates(["NPRIN", "NPARC"]))
g = two.groupby("NPRIN").P037_SS.agg(["min", "max", "size"]); g = g[g["size"] == 2]
print(f"2-parcel farmers whose two parcels are within 30% size (area can't separate them): "
      f"{(g['max'] / g['min'] < 1.3).mean():.0%}")

verified single-parcel matches: 43,888


  census P037_SS vs polygon AREA_S_HA  ·  Pearson 0.148  Spearman 0.157
  areas agree within +/-20%: 10.3%
  areas agree within +/-30%: 23.8%
2-parcel farmers whose two parcels are within 30% size (area can't separate them): 76%


**Conclusion — area does not help, so we keep the name-only match.** Even on verified
single-parcel matches the census and cadastral areas are almost uncorrelated (Pearson ≈ 0.01–0.15;
only ~10% within ±20%). `P037_SS` is a rough farmer self-estimate, and a census *parcela* (land a
farmer works) is not the same spatial unit as a PETT *predio* (titled property) — whereas the PETT
areas (polygon `AREA_S_HA`, bridge `area_ha`, SSET `AREA`) agree with *each other* at ~0.998. And
even if area were accurate, ~¾ of two-parcel farmers have parcels within 30% of each other's size,
so it couldn't separate them. Requiring area agreement would drop ~90% of true matches for no
precision gain — strictly worse. **The census↔PETT link is farmer-level, not parcel-level.** For
reliable parcel-level `(crop, year, polygon)`, use notebook 03 (PETT real keys, no census needed).

## 11 · Save outputs (lean)

* **`merged_parcels.parquet`** — one GeoDataFrame row per linked census productor (geometry + owner
  + componentised name + both crop summaries + dates + confidence).
* **`crop_records.parquet`** — long crop-record table (census + SSET), keyed by `COD_PREDIO`.

In [15]:
p1 = OUT / "merged_parcels.parquet"; p2 = OUT / "crop_records.parquet"
parcels_out.to_parquet(p1)
crop_records.to_parquet(p2, index=False)
print(f"wrote {p1}  ({p1.stat().st_size/1e6:.1f} MB · {len(parcels_out):,} rows)")
print(f"wrote {p2}  ({p2.stat().st_size/1e6:.1f} MB · {len(crop_records):,} rows)")

wrote ../data/processed/merged_parcels.parquet  (20.4 MB · 45,942 rows)
wrote ../data/processed/crop_records.parquet  (0.8 MB · 129,950 rows)


### Summary & caveats

* **The census *does* carry the farmer name** (`P009_01/02/03`); it is the only census↔PETT bridge
  (no shared codes; `N_SEA`/`COD_INEI` empty in the shapefile). We normalise the three different
  name formats into components (`name1..name4`) and match on several routes.
* **Names are not unique** — a match is candidate evidence. We quantify ambiguity and attach a
  `link_confidence`; treat `low` links as provisional. DNI is *not* on the census side, so it cannot
  disambiguate.
* **No parcel-level key exists**, so pinning the single exact polygon for a productor is uncertain
  (only ~43% exact-`COD_PREDIO` agreement with `Base`, even at high confidence) — mostly because
  multi-parcel owners and shared names let both methods pick different parcels of the same person.
  The person/district/crop link is far more reliable than the exact-parcel link.
* **Polygon names are sometimes truncated** (`RIVAS GA,`) — those parcels can't be name-matched;
  reaching them needs the SSET/catastro name (untruncated) → `CodigoSSET` → `COD_PREDIO`.
* **Independent name-matching is complementary to `Base`**, not a superset: the union of both is the
  most robust link. To exceed both, a fuzzy/edit-distance pass (blocked by UBIGEO + apellido) is the
  natural next step.
* **Two crop declarations** (census 2012 vs SSET 1998–2007) will disagree — they describe different
  years; keep both. Census coordinates are SEA-centroids (coarse) — use the polygons for location.